# 🐝 ME2 CHAT-SWARM — CONNECT-ALL: один Colab → 5 GPU-узлов роя (T4)

Один запуск на одном GPU-сеансе Colab регистрирует **все 5 моделей** как отдельные узлы роя v1.4.0:

| № | Узел | Модель | Tier |
|---|---|---|---|
| 1 | colab-1-llama31-8b | llama3.1:8b | 40 |
| 2 | colab-2-qwen3-8b | qwen3:8b | 86 |
| 3 | colab-3-gemma3-12b | gemma3:12b | 70 |
| 4 | colab-4-phi4-14b | phi4:14b | 75 |
| 5 | colab-5-mistral-nemo-12b | mistral-nemo:12b | 65 |

**Как работает:** Ollama поднимает все 5 моделей на одном сервере (VRAM менеджится автоматом, LRU выгрузка), cloudflared открывает один туннель, рой получает 5 динамических провайдеров и сам вызывает их по tier-порядку (сильные — первыми).

**Публичный адрес песочницы** берётся автоматом с GitHub (`colab/sandbox-base.json`); если там пусто — ноутбук попросит вставить его один раз.

▶️ **Запуск:** Среда выполнения → Запустить всё (Run all). Ждать ~8–12 мин на T4.

In [ ]:
# Шаг 1. Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama установлен')

In [ ]:
# Шаг 2. Запуск ollama serve
import subprocess, time, socket, os

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_KEEP_ALIVE'] = '5m'   # LRU выгрузка: VRAM держит 1-2 модели, остальные подгружаются по требованию

srv = subprocess.Popen(['ollama', 'serve'],
                       stdout=open('ollama_serve.log', 'w'),
                       stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        s = socket.create_connection(('127.0.0.1', 11434), timeout=1); s.close()
        print('✅ Ollama поднят на :11434'); break
    except OSError:
        time.sleep(1)
else:
    raise SystemExit('❌ Ollama не поднялся за 60 с — см. ollama_serve.log')

In [ ]:
# Шаг 3. Загрузка всех 5 моделей (~34 ГБ на диск, одновременно в VRAM — 1-2)
MODELS = {
    'colab-1-llama31-8b':       'llama3.1:8b',
    'colab-2-qwen3-8b':         'qwen3:8b',
    'colab-3-gemma3-12b':       'gemma3:12b',
    'colab-4-phi4-14b':         'phi4:14b',
    'colab-5-mistral-nemo-12b': 'mistral-nemo:12b',
}
for name, model in MODELS.items():
    print(f'\n📥 {model} ...')
    r = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
    print(r.stdout[-300:] if r.stdout else '', r.stderr[-200:] if r.returncode else '')
    print('✅ OK' if r.returncode == 0 else f'❌ ошибка {model}')
!ollama list

In [ ]:
# Шаг 4. Самотест: один звонок в головную модель роя (qwen3:8b)
import json, urllib.request, time

def chat(prompt, model='qwen3:8b', num_predict=64):
    body = json.dumps({'model': model, 'stream': False,
                       'messages': [{'role': 'user', 'content': prompt}],
                       'options': {'num_predict': num_predict}}).encode()
    t0 = time.time()
    req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=body,
                                 headers={'Content-Type': 'application/json'})
    d = json.loads(urllib.request.urlopen(req, timeout=180).read())
    return d['message']['content'][:120], round(time.time() - t0, 1)

txt, sec = chat('Ответь одним словом: ты живой?')
print(f'✅ самотест {sec}s → {txt!r}')

In [ ]:
# Шаг 5. Туннель cloudflared (quick, без аккаунта)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                       stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
TUNNEL_URL = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        TUNNEL_URL = m.group(0); break
print('TUNNEL_URL =', TUNNEL_URL if TUNNEL_URL else '❌ не получен — см. tunnel.log')

In [ ]:
# Шаг 6. PUBLIC_BASE: сначала GitHub, если пусто — одно вставление вручную
import json, urllib.request

PUBLIC_BASE = ''
try:
    u = 'https://raw.githubusercontent.com/PatrickFrome/Compute/context-vault/colab/sandbox-base.json'
    PUBLIC_BASE = json.loads(urllib.request.urlopen(u, timeout=15).read()).get('publicBase', '')
    if PUBLIC_BASE:
        print('✅ PUBLIC_BASE забран с GitHub:', PUBLIC_BASE)
except Exception as e:
    print('⚠️ GitHub-забор не удался:', e)

if not PUBLIC_BASE:
    PUBLIC_BASE = input('Вставьте публичный адрес песочницы (кнопка Open in New Tab, напр. https://xxxx.z.ai): ').strip().rstrip('/')
print('PUBLIC_BASE =', PUBLIC_BASE)

In [ ]:
# Шаг 7. РЕГИСТРАЦИЯ всех 5 узлов у роя
import json, urllib.request, time

def register(public_base, tunnel_url, model, name):
    body = json.dumps({'url': tunnel_url, 'model': model, 'name': name}).encode()
    req = urllib.request.Request(
        f'{public_base.rstrip("/")}/colab/register?XTransformPort=3046',
        data=body, headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=60).read())

ok = 0
if TUNNEL_URL and PUBLIC_BASE and not PUBLIC_BASE.startswith(('PASTE', 'Вставь')):
    for name, model in MODELS.items():
        try:
            r = register(PUBLIC_BASE, TUNNEL_URL, model, name)
            print(f"{'\u2705' if r.get('ok') else '\u274c'} {name} ({model}): {json.dumps(r, ensure_ascii=False)[:200]}")
            ok += 1 if r.get('ok') else 0
            time.sleep(1)
        except Exception as e:
            print(f'❌ {name}: {e}')
    print(f'\n🔥 ПОДКЛЮЧЕНО {ok}/5 узлов — рой уже работает через этот GPU' if ok else '\n❌ ни один узел не зарегистрирован')
    print('Проверка: GET {PUBLIC_BASE}/colab/status?XTransformPort=3046')
else:
    print(f'⚠️ Заполните PUBLIC_BASE в шаге 6 и повторите. TUNNEL_URL = {TUNNEL_URL}')

In [ ]:
# Шаг 8. Жизнь сеанса: перерегистрация каждые 5 мин (tunnel тоже обновляется если умрет)
import threading, time, re, subprocess

def keepalive():
    global TUNNEL_URL
    while True:
        time.sleep(300)
        try:
            m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
            if m and m.group(0) != TUNNEL_URL:
                TUNNEL_URL = m.group(0)
                print('♻️ туннель обновился:', TUNNEL_URL)
            for name, model in MODELS.items():
                try:
                    register(PUBLIC_BASE, TUNNEL_URL, model, name)
                except Exception:
                    pass
            print('♻️ перерегистрация 5 узлов OK', time.strftime('%H:%M:%S'))
        except Exception as e:
            print('⚠️ keepalive:', e)

if TUNNEL_URL and PUBLIC_BASE:
    threading.Thread(target=keepalive, daemon=True).start()
    print('✅ keepalive запущен. Вкладку Colab держать открытой.')
else:
    print('⚠️ сначала завершите шаги 5–7')